# 13_FHIR_All_Resources_Validation

Validates Bronze, Silver, SCD Type 2 history, and Gold tables for Patient, Encounter, Observation, and Condition. Run separately after the pipeline finishes. This notebook reads existing tables only.

In [1]:
from pyspark.sql import functions as F

resources = [
    {"name":"Patient","bronze_candidates":["bronze.patient","bronze.fhir_patient"],"silver":"silver.patient","history":"silver.patient_history","gold":"gold.dim_patient","gold_keys":["patient_id","id"]},
    {"name":"Encounter","bronze_candidates":["bronze.encounter","bronze.fhir_encounter"],"silver":"silver.encounter","history":"silver.encounter_history","gold":"gold.fact_encounter","gold_keys":["encounter_id","id"]},
    {"name":"Observation","bronze_candidates":["bronze.observation","bronze.fhir_observation"],"silver":"silver.observation","history":"silver.observation_history","gold":"gold.fact_observation","gold_keys":["observation_id","id"]},
    {"name":"Condition","bronze_candidates":["bronze.condition","bronze.fhir_condition"],"silver":"silver.condition","history":"silver.condition_history","gold":"gold.fact_condition","gold_keys":["condition_id","id"]}
]

def table_exists(name):
    try:
        return spark.catalog.tableExists(name)
    except Exception:
        return False

def first_existing(candidates):
    return next((name for name in candidates if table_exists(name)), None)

print("Validation setup complete:", len(resources), "resources")


StatementMeta(, fcdda943-f00e-4e7c-b7a3-7dc4fd6897b9, 3, Finished, Available, Finished, False)

Validation setup complete: 4 resources


In [2]:
# TABLE AVAILABILITY AND ROW COUNTS
summary = []

for r in resources:
    print(f"\\n===== {r['name'].upper()} =====")
    bronze = first_existing(r["bronze_candidates"])
    checks = [("Bronze", bronze), ("Silver", r["silver"]), ("History", r["history"]), ("Gold", r["gold"])]

    for layer, table_name in checks:
        if table_name is None:
            expected = ", ".join(r["bronze_candidates"])
            summary.append((r["name"], layer, expected, None, "MISSING", "No candidate Bronze table found"))
            print(f"{layer}: MISSING; tried {expected}")
        elif not table_exists(table_name):
            summary.append((r["name"], layer, table_name, None, "MISSING", "Table does not exist"))
            print(f"{layer} ({table_name}): MISSING")
        else:
            try:
                n = spark.table(table_name).count()
                summary.append((r["name"], layer, table_name, n, "SUCCESS", ""))
                print(f"{layer} ({table_name}): {n:,} rows")
            except Exception as e:
                msg = str(e).splitlines()[0][:250]
                summary.append((r["name"], layer, table_name, None, "ERROR", msg))
                print(f"{layer} ({table_name}): ERROR - {msg}")

summary_df = spark.createDataFrame(summary, ["resource","layer","table_name","row_count","status","error_message"])
display(summary_df.orderBy("resource","layer"))


StatementMeta(, fcdda943-f00e-4e7c-b7a3-7dc4fd6897b9, 4, Finished, Available, Finished, False)

\n===== PATIENT =====
Bronze (bronze.patient): 1,673 rows
Silver (silver.patient): 1,670 rows
History (silver.patient_history): 2,067 rows
Gold (gold.dim_patient): 1,670 rows
\n===== ENCOUNTER =====
Bronze (bronze.encounter): 2,074 rows
Silver (silver.encounter): 2,074 rows
History (silver.encounter_history): 2,106 rows
Gold (gold.fact_encounter): 2,074 rows
\n===== OBSERVATION =====
Bronze (bronze.observation): 28,286 rows
Silver (silver.observation): 28,286 rows
History (silver.observation_history): 64,348 rows
Gold (gold.fact_observation): 28,286 rows
\n===== CONDITION =====
Bronze (bronze.condition): 12,013 rows
Silver (silver.condition): 12,013 rows
History (silver.condition_history): 13,589 rows
Gold (gold.fact_condition): 12,013 rows


SynapseWidget(Synapse.DataFrame, e453b69a-2899-492f-91f7-a6d43ac16f30)

In [3]:
# DATA QUALITY CHECKS
quality = []

for r in resources:
    print(f"\\n===== {r['name'].upper()} =====")

    # Silver duplicate IDs
    if table_exists(r["silver"]):
        try:
            df = spark.table(r["silver"])
            if "id" in df.columns:
                n = df.groupBy("id").count().filter(F.col("count") > 1).count()
                quality.append((r["name"], "Silver duplicate IDs", n, "PASS" if n == 0 else "CHECK", ""))
                print("Silver duplicate IDs:", n)
            else:
                quality.append((r["name"], "Silver duplicate IDs", None, "SKIPPED", "Column id not found"))
        except Exception as e:
            quality.append((r["name"], "Silver duplicate IDs", None, "ERROR", str(e).splitlines()[0][:250]))
    else:
        quality.append((r["name"], "Silver duplicate IDs", None, "SKIPPED", "Silver table missing"))

    # Gold duplicate key
    if table_exists(r["gold"]):
        try:
            df = spark.table(r["gold"])
            key = next((c for c in r["gold_keys"] if c in df.columns), None)
            if key:
                n = df.groupBy(key).count().filter(F.col("count") > 1).count()
                quality.append((r["name"], f"Gold duplicate {key}", n, "PASS" if n == 0 else "CHECK", ""))
                print(f"Gold duplicate {key}:", n)
            else:
                quality.append((r["name"], "Gold duplicate key", None, "SKIPPED", "Expected key column not found"))
        except Exception as e:
            quality.append((r["name"], "Gold duplicate key", None, "ERROR", str(e).splitlines()[0][:250]))
    else:
        quality.append((r["name"], "Gold duplicate key", None, "SKIPPED", "Gold table missing"))

    # SCD Type 2 current-version uniqueness
    if table_exists(r["history"]):
        try:
            df = spark.table(r["history"])
            missing = [c for c in ["id", "is_current"] if c not in df.columns]
            if not missing:
                n = (df.filter(F.col("is_current") == True).groupBy("id")
                       .count().filter(F.col("count") > 1).count())
                quality.append((r["name"], "Multiple current history versions", n, "PASS" if n == 0 else "CHECK", ""))
                print("IDs with multiple current history versions:", n)
            else:
                quality.append((r["name"], "SCD2 current-version check", None, "SKIPPED", "Missing columns: " + ", ".join(missing)))
        except Exception as e:
            quality.append((r["name"], "SCD2 current-version check", None, "ERROR", str(e).splitlines()[0][:250]))
    else:
        quality.append((r["name"], "SCD2 current-version check", None, "SKIPPED", "History table missing"))

quality_df = spark.createDataFrame(quality, ["resource","check_name","issue_count","result","details"])
display(quality_df.orderBy("resource","check_name"))


StatementMeta(, fcdda943-f00e-4e7c-b7a3-7dc4fd6897b9, 5, Finished, Available, Finished, False)

\n===== PATIENT =====
Silver duplicate IDs: 0
Gold duplicate patient_id: 0
IDs with multiple current history versions: 0
\n===== ENCOUNTER =====
Silver duplicate IDs: 0
Gold duplicate encounter_id: 0
IDs with multiple current history versions: 0
\n===== OBSERVATION =====
Silver duplicate IDs: 0
Gold duplicate observation_id: 0
IDs with multiple current history versions: 0
\n===== CONDITION =====
Silver duplicate IDs: 0
Gold duplicate condition_id: 0
IDs with multiple current history versions: 0


SynapseWidget(Synapse.DataFrame, 820004ba-b5f2-4f2e-8dbf-45c89cad4a80)

## Interpreting results

- `SUCCESS`: table found and row count read.
- `MISSING`: expected table not found; verify the actual table name in the corresponding notebook.
- `PASS`: no duplicates were detected by that check.
- `CHECK`: duplicates were detected and need review.
- `SKIPPED`: table or expected column was unavailable.
- `ERROR`: inspect the details column.

These checks do not prove that every historical change at the source was captured.